# Incomplete Code

### Print MT Model Architecture  

In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

# Load the tokenizer and model
tokenizer_mt = AutoTokenizer.from_pretrained("Bohanlu/Taigi-Llama-2-Translator-7B")
model_mt = AutoModel.from_pretrained("Bohanlu/Taigi-Llama-2-Translator-7B", trust_remote_code=True)

print(model_mt)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/813k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

LlamaModel(
  (embed_tokens): Embedding(56024, 4096)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
        (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
        (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((4096,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)


### Modify model to add a classification head after last decoder layer

In [8]:
import torch
import torch.nn as nn
from transformers import LlamaModel

class LlamaForTokenClassification(nn.Module):
    def __init__(self, pretrained_model_name, num_labels=2):
        super().__init__()
        self.model = LlamaModel.from_pretrained(pretrained_model_name)
        self.classifier = nn.Linear(self.model.config.hidden_size, num_labels)  # (4096 → 2)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # Shape: (batch_size, seq_len, 4096)
        logits = self.classifier(hidden_states)  # Shape: (batch_size, seq_len, num_labels)
        return logits


In [20]:
num_labels = 3

In [14]:
model_sl = LlamaForTokenClassification(pretrained_model_name="Bohanlu/Taigi-Llama-2-Translator-7B", num_labels=num_labels) # can also choose a larger model "Bohanlu/Taigi-Llama-2-Translator-13B" for better performance

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [17]:
tokenizer_sl = AutoTokenizer.from_pretrained("Bohanlu/Taigi-Llama-2-Translator-7B")

### Tokenize and Align Labels

In [9]:
# TODO create the training data format
def align_labels_with_tokens(labels, word_ids):
  pass

In [ ]:
from transformers import AutoTokenizer

# Example
sentence = "猶_@ 是_@ 被_@ 警_@ 方_@ 移 送 法 辦 。"
labels = ["H", "H", "H", "H", "H", "C", "C", "C", "C", "C", "O"]

# Convert labels to numerical format
label_map = {"C": 0, "H": 1, "O": 2}  # Add 'O' for non-language tokens
label_ids = [label_map[label] for label in labels]

# Tokenize while aligning labels
tokens = tokenizer_sl(sentence, return_offsets_mapping=True, truncation=True)
aligned_labels = align_labels_with_tokens(tokens, label_ids)


### Training Setup

In [19]:
# TODO code up train_dataloader
train_dataloader = None

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_mt.parameters(), lr=5e-5)

for batch in train_dataloader:
    input_ids, labels = batch["input_ids"], batch["labels"]

    optimizer.zero_grad()
    logits = model_sl(input_ids)

    # Compute loss
    loss = criterion(logits.view(-1, num_labels), labels.view(-1))
    loss.backward()

    optimizer.step()


### Optional Training
- Pretrain using masked language modeling on unsupervised code-switched data
- targeted fine-tuning on high-error tokens
